# Tier 1 Score Generation — INFERENCE ONLY

This notebook **no longer trains** any Tier-1 model. It loads the four **pretrained** Week-2 MLP agents selected for Tier-2 fusion and runs **inference only** on the held-out validation and test users.

| Modality | Model | Undersample |
|---|---|---|
| HTTP | MLP (pretrained) | 6:1 |
| File/Device | MLP (pretrained) | 4:1 |
| Email | MLP (pretrained) | 4:1 |
| Logon | MLP (pretrained) | 4:1 |

The old Logon **UniLSTM** is removed. Outputs remain `week3/tier1_validation_scores.csv` and `week3/tier1_test_scores.csv` with schema `user, week, day, label, logon_score, email_score, file_device_score, http_score`.

## 1. Imports, Seed & Configuration

In [1]:
import random
import sys
from pathlib import Path
import os, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# === Make week3 importable from any kernel CWD ===
# Walk up from the working directory to find the project root
# (pyproject.toml anchor, project portability convention) and put it on
# sys.path so `from week3 import tier1_lib` always resolves.
_curr = Path.cwd()
while not (_curr / 'pyproject.toml').exists() and _curr != _curr.parent:
    _curr = _curr.parent
PROJECT_ROOT = _curr
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from week3 import tier1_lib as T
except ModuleNotFoundError:
    # Fallback: if only the week3/ directory itself is on sys.path
    # (e.g. kernel CWD is week3/), import the module directly.
    from tier1_lib import T as tier1_lib
    import tier1_lib as T  # noqa: F811 (alias for API parity)

# === Determinism: seed all RNGs (single source: tier1_lib) ===
SEED = T.SEED
T.seed_all(SEED)

print(f"Device: {T.DEVICE}")
print(f"Project root: {PROJECT_ROOT}")


Device: cpu
Project root: C:\insider_threat_detection_project2


C:\insider_threat_detection_project2\.venv\Lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


## 2. Canonical Tier-1 Artifacts (tier1/<modality>/)

Loaded from `tier1/http|file_device|email|logon/{model.pt, scaler.joblib, meta.json}` (single source of truth).

In [2]:
# Canonical finalized artifacts (single source of truth: tier1/<modality>/)
MODALITIES = T.MODALITIES
BUNDLES = T.load_all_artifacts()
for mod, (model, scaler, feature_cols, meta) in BUNDLES.items():
    print(f"  {mod:<12} ratio={meta['sampling_ratio']} input_dim={meta['input_dim']} "
          f"feature_cols={len(feature_cols)} scaler_mean={len(scaler.mean_)} "
          f"scaler_type={meta['scaler_type']}")


  http         ratio=6:1 input_dim=204 feature_cols=204 scaler_mean=204 scaler_type=StandardScaler
  file_device  ratio=4:1 input_dim=215 feature_cols=215 scaler_mean=215 scaler_type=StandardScaler
  email        ratio=4:1 input_dim=47 feature_cols=47 scaler_mean=47 scaler_type=StandardScaler
  logon        ratio=4:1 input_dim=20 feature_cols=20 scaler_mean=20 scaler_type=StandardScaler


## 3. Model Architecture (`tier1_lib.ModalityFFNN`, exact Week-2 MLP)

In [3]:
# Model architecture: tier1_lib.ModalityFFNN (exact Week-2 MLP)
print(T.ModalityFFNN.__doc__)
assert all(meta['arch'] == 'ModalityFFNN' and meta['model_type'] == 'MLP'
           for _, _, _, meta in BUNDLES.values()), "all Tier-1 agents must be MLPs"


3-layer MLP: input -> 64 -> 32 -> 1 (ReLU, dropout 0.2).


## 4. Load the Pre-Split Data (train/val/test users)

In [4]:
for f in ['dayr4.2-percentile30-train.pkl', 'dayr4.2-percentile30-validation.pkl', 'dayr4.2-percentile30-test.pkl']:
    if not (T.SPLIT_DIR / f).exists():
        raise FileNotFoundError(f"Split file missing: {T.SPLIT_DIR / f}")

train_df = pd.read_pickle(str(T.SPLIT_DIR / 'dayr4.2-percentile30-train.pkl'))
val_df   = pd.read_pickle(str(T.SPLIT_DIR / 'dayr4.2-percentile30-validation.pkl'))
test_df  = pd.read_pickle(str(T.SPLIT_DIR / 'dayr4.2-percentile30-test.pkl'))

for dfp in (train_df, val_df, test_df):
    for col in dfp.select_dtypes(include=['float64']).columns:
        dfp[col] = dfp[col].astype('float32')

split_info = T.verify_split(train_df, val_df, test_df)
print(f"Split verified: {split_info}")
train_users = set(train_df['user'].unique())
val_users = set(val_df['user'].unique())
test_users = set(test_df['user'].unique())
print(f"Train rows: {len(train_df)} ({len(train_users)} users)")
print(f"Val   rows: {len(val_df)} ({len(val_users)} users)")
print(f"Test  rows: {len(test_df)} ({len(test_users)} users)")


Split verified: {'train_users': 600, 'val_users': 200, 'test_users': 200, 'malicious': {'train': 42, 'val': 14, 'test': 14}}
Train rows: 185744 (600 users)
Val   rows: 61076 (200 users)
Test  rows: 61203 (200 users)


## 5. Feature Selection (deterministic, from `tier1_lib`, no `set()`)

The exact ordered feature list is persisted in each `meta.json`; the notebook asserts its reconstruction matches the saved order.

In [5]:
# Deterministic feature columns (sorted, no set()) — from tier1_lib
FEAT_COLS = {m: T.get_modality_columns(train_df.columns.tolist(), m)
             for m in ['http', 'file_device', 'email', 'logon']}
for m, cols in FEAT_COLS.items():
    meta_cols = BUNDLES[m][3]['feature_cols']
    assert cols == meta_cols, f"{m}: notebook feature order != saved meta order"
    print(f"  {m:<12} features={len(cols)} (matches saved meta: {cols == meta_cols})")


  http         features=204 (matches saved meta: True)
  file_device  features=215 (matches saved meta: True)
  email        features=47 (matches saved meta: True)
  logon        features=20 (matches saved meta: True)


## 6. Preprocessing — Scaler (LOADED, never refit)

Scalers were fit on **training users only (60%)** during retrain and persisted to `tier1/<mod>/scaler.joblib`. The notebook loads them.

In [6]:
# Scalers are LOADED from tier1/<mod>/scaler.joblib (fitted on TRAIN users only at retrain).
SCALERS = {m: BUNDLES[m][1] for m in MODALITIES}
for m in MODALITIES:
    assert len(SCALERS[m].mean_) == len(FEAT_COLS[m]), f"{m}: scaler/feature len mismatch"
print("Scalers loaded from persisted artifacts (not refit).")


Scalers loaded from persisted artifacts (not refit).


## 7. Inference Only (validation + test)

Uses persisted model + scaler + feature order. No training, no scaler refit, no undersampling at inference.

In [7]:
# Inference only: score with persisted model + scaler + feature order
scores_val = {}
scores_test = {}
for m in MODALITIES:
    model, scaler, feature_cols, meta = BUNDLES[m]
    s_val = T.score_dataframe(val_df, model, scaler, feature_cols)
    s_test = T.score_dataframe(test_df, model, scaler, feature_cols)
    scores_val[m] = s_val
    scores_test[m] = s_test
    print(f"  {m:<12} val={len(s_val)} test={len(s_test)}")


  http         val=61076 test=61203
  file_device  val=61076 test=61203
  email        val=61076 test=61203
  logon        val=61076 test=61203


## 8. Merge & Align Four Modality Scores

Outer-merge on `(user, week, day, label)` across the four modalities. Since Logon is now a **daily MLP** (score for every user/day, like the others), no forward-fill is needed; missing cells (if any) are filled with 0 to keep the exact downstream schema.

In [8]:
def build_tier1(scores, dfp):
    out = dfp[['user', 'week', 'day']].copy().reset_index(drop=True)
    out['label'] = (dfp['insider'].values > 0).astype(int)
    for m in ['logon', 'email', 'file_device', 'http']:
        out[f'{m}_score'] = scores[m]
    return out[['user', 'week', 'day', 'label',
                'logon_score', 'email_score', 'file_device_score', 'http_score']]

tier1_val = build_tier1(scores_val, val_df).sort_values(['user', 'week', 'day']).reset_index(drop=True)
tier1_test = build_tier1(scores_test, test_df).sort_values(['user', 'week', 'day']).reset_index(drop=True)
print(f"tier1_val shape : {tier1_val.shape}")
print(f"tier1_test shape: {tier1_test.shape}")
tier1_test.head()


tier1_val shape : (61076, 8)
tier1_test shape: (61203, 8)


,user,week,day,label,logon_score,email_score,file_device_score,http_score
0,5.0,5.0,37.0,0,0.277578,0.131737,0.014935,0.003352
1,5.0,5.0,38.0,0,0.277578,0.172129,0.014935,0.003738
2,5.0,5.0,39.0,0,0.277578,0.191476,0.014935,0.001489
3,5.0,6.0,42.0,0,0.277578,0.264323,0.026009,0.003609
4,5.0,6.0,43.0,0,0.277578,0.301204,0.026009,0.001346


## 9. Save Tier-1 Scores (validation + test)

In [9]:
val_path = T.PROJECT_ROOT / 'week3' / 'tier1_validation_scores.csv'
test_path = T.PROJECT_ROOT / 'week3' / 'tier1_test_scores.csv'
tier1_val.to_csv(val_path, index=False)
tier1_test.to_csv(test_path, index=False)
print(f"Saved {val_path.name}: {len(tier1_val)} rows | {tier1_val['user'].nunique()} users | {tier1_val['label'].sum()} positive rows")
print(f"Saved {test_path.name}: {len(tier1_test)} rows | {tier1_test['user'].nunique()} users | {tier1_test['label'].sum()} positive rows")


Saved tier1_validation_scores.csv: 61076 rows | 200 users | 191 positive rows
Saved tier1_test_scores.csv: 61203 rows | 200 users | 201 positive rows


## 10. Sanity Checks (schema, ranges, users, leakage)

In [10]:
cols = ['user', 'week', 'day', 'label', 'logon_score', 'email_score', 'file_device_score', 'http_score']
assert list(tier1_val.columns) == cols, f"val cols mismatch: {list(tier1_val.columns)}"
assert list(tier1_test.columns) == cols, f"test cols mismatch: {list(tier1_test.columns)}"

for name, t1 in [('VALIDATION', tier1_val), ('TEST', tier1_test)]:
    sc = t1[['logon_score', 'email_score', 'file_device_score', 'http_score']]
    print(f"[{name}] all numeric: {sc.apply(pd.to_numeric, errors='coerce').notna().all().all()}")
    print(f"[{name}] finite:     {np.isfinite(sc.values).all()}")
    print(f"[{name}] range [0,1]: {((sc.values >= 0) & (sc.values <= 1)).all()}")
    print(f"[{name}] NaNs:       {int(sc.isna().sum().sum())}")

assert set(tier1_val['user'].unique()) == val_users, "val file contains non-val users!"
assert set(tier1_test['user'].unique()) == test_users, "test file contains non-test users!"
print("[OK] Validation file has only validation users; Test file has only test users.")
print(f"[OK] Val ∩ Test = {len(set(tier1_val['user'].unique()) & set(tier1_test['user'].unique()))}")


[VALIDATION] all numeric: True
[VALIDATION] finite:     True
[VALIDATION] range [0,1]: True
[VALIDATION] NaNs:       0
[TEST] all numeric: True
[TEST] finite:     True
[TEST] range [0,1]: True
[TEST] NaNs:       0
[OK] Validation file has only validation users; Test file has only test users.
[OK] Val ∩ Test = 0


## 11. Final Report

In [11]:
print("=" * 60)
print("TIER-1 PRETRAINED MODELS USED FOR TIER-2")
print("=" * 60)
print("HTTP        : MLP 6:1")
print("File/Device : MLP 4:1")
print("Email       : MLP 4:1")
print("Logon       : MLP 4:1")
print("\nMode        : INFERENCE ONLY")
print("Training    : DISABLED (artifacts loaded from tier1/<modality>/)")
print(f"Seed        : {SEED}")
print("=" * 60)
print(f"Validation users: {len(val_users)}")
print(f"Test users:       {len(test_users)}")
print(f"Validation score shape: {tier1_val.shape}")
print(f"Test score shape:       {tier1_test.shape}")
print("=" * 60)


TIER-1 PRETRAINED MODELS USED FOR TIER-2
HTTP        : MLP 6:1
File/Device : MLP 4:1
Email       : MLP 4:1
Logon       : MLP 4:1

Mode        : INFERENCE ONLY
Training    : DISABLED (artifacts loaded from tier1/<modality>/)
Seed        : 42
Validation users: 200
Test users:       200
Validation score shape: (61076, 8)
Test score shape:       (61203, 8)
